# CARE-BED — final preprocessing-filter ablation on A13 + B12

This notebook repeats the preprocessing-filter ablation on the **final 25-recording CARE-BED dataset**:

```text
Environment A: 13 historical recordings
Environment B: 12 recordings = s01, s02, s03 for each activity
TOTAL:         25 independent recordings
```

The temporal window is fixed at **20 CSI frames** and only the filtering configuration changes:

```text
W20_NONE = no filtering
W20_H    = Hampel only
W20_SG   = Savitzky–Golay only
W20_HSG  = Hampel + Savitzky–Golay
```

All four configurations use the **same fixed 5-fold recording-disjoint split**, the same BiLSTM architecture and training settings, stride 5, and a `StandardScaler` fitted on TRAIN data only.

The experiment is designed specifically to answer Reviewer #1's request to repeat the filter ablation under the corrected recording-disjoint protocol. It does **not** retrain or replace the final deployment model automatically; it only evaluates the effect of preprocessing filters under a controlled protocol.


## 0. Install extra packages

In [ ]:
!pip -q install hampel joblib

## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
from math import sqrt
import csv
import json
import random
import re
import shutil
import time
import zipfile

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hampel import hampel
from scipy.signal import savgol_filter

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## 2. Upload and extract the two ZIP files

In [ ]:
WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()

OLD_ZIP_PATH = WORKDIR / "envA_dataset.zip"
NEW_ZIP_PATH = WORKDIR / "carebed_new_data.zip"

if not OLD_ZIP_PATH.exists() or not NEW_ZIP_PATH.exists():
    try:
        from google.colab import files
        print("Upload envA_dataset.zip and carebed_new_data.zip")
        files.upload()
    except ImportError:
        pass

if not OLD_ZIP_PATH.exists():
    raise FileNotFoundError("Missing envA_dataset.zip")

if not NEW_ZIP_PATH.exists():
    raise FileNotFoundError("Missing carebed_new_data.zip")

OLD_ROOT = WORKDIR / "envA_dataset"
NEW_ROOT = WORKDIR / "carebed_new_data"

for root in [OLD_ROOT, NEW_ROOT]:
    if root.exists():
        shutil.rmtree(root)

with zipfile.ZipFile(OLD_ZIP_PATH, "r") as zf:
    zf.extractall(WORKDIR)

with zipfile.ZipFile(NEW_ZIP_PATH, "r") as zf:
    zf.extractall(WORKDIR)

if not OLD_ROOT.exists():
    raise FileNotFoundError(
        "envA_dataset.zip must contain a top-level envA_dataset/ folder."
    )

if not NEW_ROOT.exists():
    raise FileNotFoundError(
        "carebed_new_data.zip must contain a top-level carebed_new_data/ folder."
    )

print("Environment A files:")
for path in sorted(OLD_ROOT.iterdir()):
    if path.is_file():
        print(" ", path.name)

print("\nEnvironment B folders:")
for path in sorted(NEW_ROOT.iterdir()):
    if path.is_dir():
        print(" ", path.name)

## 3. Configuration

In [ ]:
EXPECTED_ROLE = "AP"
NEW_MAC = "24:0A:C4:02:29:F0"

OLD_SOURCE_TOKENS = {
    "inactivity": "br_ak",
    "lying down": "lezenie",
    "sitting up": "siedzenie",
    "fidgeting": "wiercenie",
}

# Historical Environment A recording boundaries: 1-based, inclusive.
OLD_RECORDING_RANGES = {
    "inactivity": [
        ("A_inactivity_s01", 1, 1765),
        ("A_inactivity_s02", 1766, 2427),
        ("A_inactivity_s03", 2428, 3095),
        ("A_inactivity_s04", 3096, 3202),
    ],
    "lying down": [
        ("A_lying_down_s01", 1, 607),
        ("A_lying_down_s02", 608, 1779),
        ("A_lying_down_s03", 1780, 2947),
    ],
    "sitting up": [
        ("A_sitting_up_s01", 1, 601),
        ("A_sitting_up_s02", 602, 1739),
        ("A_sitting_up_s03", 1740, 2922),
    ],
    "fidgeting": [
        ("A_fidgeting_s01", 1, 1123),
        ("A_fidgeting_s02", 1124, 1810),
        ("A_fidgeting_s03", 1811, 2955),
    ],
}

EXPECTED_OLD_LINE_COUNTS = {
    "inactivity": 3202,
    "lying down": 2947,
    "sitting up": 2922,
    "fidgeting": 2955,
}

B_FOLDERS = {
    "inactivity": "inactivity",
    "lying down": "lying_down",
    "sitting up": "sitting_up",
    "fidgeting": "fidgeting",
}

VALID_B_SESSIONS = {1, 2, 3}

FIRST_CSI_VALUES = 128
RAW_FEATURES = 64

COLUMNS_TO_DROP = [2, 3, 4, 5, 32, 59, 60, 61, 62, 63]
N_FEATURES = 54

STRIDE = 5
DEV_TRAIN_FRACTION = 0.80
MIN_FRAMES = 40

HAMPEL_WINDOW = 10
SAVGOL_WINDOW = 10
SAVGOL_POLYORDER = 3

LEARNING_RATE = 0.0005
BATCH_SIZE = 8
MAX_EPOCHS = 50
PATIENCE = 5

N_SPLITS = 5

CONFIGURATIONS = [
    {"config_id": "W20_NONE", "window": 20, "use_hampel": False, "use_savgol": False, "label": "20 | no filtering"},
    {"config_id": "W20_H",    "window": 20, "use_hampel": True,  "use_savgol": False, "label": "20 | Hampel only"},
    {"config_id": "W20_SG",   "window": 20, "use_hampel": False, "use_savgol": True,  "label": "20 | Savitzky-Golay only"},
    {"config_id": "W20_HSG",  "window": 20, "use_hampel": True,  "use_savgol": True,  "label": "20 | Hampel + SG"},
]
PREPROCESS_CACHE = WORKDIR / "carebed_A13_B12_filter_ablation_preprocessed_partitions.joblib"

ALL_FOLDS_CSV = WORKDIR / "carebed_A13_B12_filter_ablation_fold_metrics.csv"
SUMMARY_CSV = WORKDIR / "carebed_A13_B12_filter_ablation_summary.csv"
ENV_SUMMARY_CSV = WORKDIR / "carebed_A13_B12_filter_ablation_environment_summary.csv"
FILTER_CSV = WORKDIR / "carebed_A13_B12_filter_ablation_article_table.csv"
PREDICTIONS_CSV = WORKDIR / "carebed_A13_B12_filter_ablation_predictions.csv"
FILTER_PDF = WORKDIR / "carebed_A13_B12_filter_ablation.pdf"
RESULT_ZIP = WORKDIR / "CARE_BED_A13_B12_filter_ablation_results.zip"

assert RAW_FEATURES - len(COLUMNS_TO_DROP) == N_FEATURES

## 4. CSI parsers

In [ ]:
def parse_bracket_payload(line):
    try:
        content = re.findall(r"\[(.*)\]", line)[0]
        return [int(x) for x in content.split(" ") if x != ""]
    except Exception:
        return None


def amplitude_from_payload(payload):
    if payload is None or len(payload) < FIRST_CSI_VALUES:
        return None

    raw = payload[:FIRST_CSI_VALUES]
    imaginary = raw[0::2]
    real = raw[1::2]

    if len(real) != RAW_FEATURES:
        return None

    return np.asarray(
        [
            sqrt(imaginary[i] ** 2 + real[i] ** 2)
            for i in range(len(real))
        ],
        dtype=np.float32,
    )


def old_line_to_amplitude(line):
    # Historical A parser: same direct bracket-payload idea as the old notebook.
    return amplitude_from_payload(
        parse_bracket_payload(line)
    )


def new_line_to_amplitude(line):
    # Current B parser: validate the new CSV structure, MAC and payload length.
    try:
        row = next(csv.reader([line]))
    except Exception:
        return None

    if len(row) != 26:
        return None
    if row[0].strip() != "CSI_DATA":
        return None
    if row[1].strip() != EXPECTED_ROLE:
        return None
    if row[2].strip().upper() != NEW_MAC.upper():
        return None

    try:
        declared_len = int(row[24].strip())
    except ValueError:
        return None

    payload = parse_bracket_payload(line)

    if payload is None or len(payload) != declared_len:
        return None

    return amplitude_from_payload(payload)


def session_number_from_name(path):
    match = re.search(
        r"_s(\d+)\.csv$",
        path.name,
        flags=re.IGNORECASE,
    )
    return int(match.group(1)) if match else None

## 5. Load Environment A (13) + final Environment B subset (12)


In [ ]:
recordings_A = []
recordings_B = []

# -----------------------
# Environment A
# -----------------------
old_source_files = {}

for activity, token in OLD_SOURCE_TOKENS.items():
    matches = [
        path for path in OLD_ROOT.glob("*.csv")
        if token in path.name.lower()
    ]

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one Environment A CSV containing '{token}', "
            f"found {[p.name for p in matches]}"
        )

    old_source_files[activity] = matches[0]


for activity, ranges in OLD_RECORDING_RANGES.items():
    source = old_source_files[activity]

    with source.open("r", encoding="utf-8", errors="ignore") as fh:
        lines = fh.readlines()

    expected = EXPECTED_OLD_LINE_COUNTS[activity]

    if len(lines) != expected:
        raise ValueError(
            f"{source.name}: expected {expected} lines, found {len(lines)}"
        )

    for recording_id, start1, end1 in ranges:
        amplitudes = [
            old_line_to_amplitude(line)
            for line in lines[start1 - 1:end1]
        ]

        amplitudes = [
            value for value in amplitudes
            if value is not None
        ]

        raw = pd.DataFrame(amplitudes)

        if raw.shape[1] != RAW_FEATURES:
            raise ValueError(
                f"{recording_id}: expected 64 features, got {raw.shape}"
            )

        recordings_A.append({
            "recording_id": recording_id,
            "label": activity,
            "environment": "Environment A",
            "frames": len(raw),
            "source_file": source.name,
            "raw": raw.reset_index(drop=True),
        })


# -----------------------
# Environment B
# -----------------------
for activity, folder_name in B_FOLDERS.items():
    folder = NEW_ROOT / folder_name

    if not folder.exists():
        raise FileNotFoundError(
            f"Missing Environment B folder: {folder}"
        )

    for path in sorted(folder.glob("*.csv")):
        if path.name.endswith("_timing.csv"):
            continue

        session_number = session_number_from_name(path)

        if session_number not in VALID_B_SESSIONS:
            continue

        amplitudes = []

        with path.open("r", encoding="utf-8", errors="ignore") as fh:
            for line in fh:
                value = new_line_to_amplitude(line)
                if value is not None:
                    amplitudes.append(value)

        if not amplitudes:
            raise ValueError(
                f"{path.name}: no valid Environment B CSI frames."
            )

        raw = pd.DataFrame(amplitudes)

        if raw.shape[1] != RAW_FEATURES:
            raise ValueError(
                f"{path.name}: expected 64 features, got {raw.shape}"
            )

        recordings_B.append({
            "recording_id": "B_" + path.stem.replace("envb_", ""),
            "label": activity,
            "environment": "Environment B",
            "frames": len(raw),
            "source_file": path.name,
            "raw": raw.reset_index(drop=True),
        })


recordings = recordings_A + recordings_B

recording_summary = pd.DataFrame([
    {
        "recording_id": r["recording_id"],
        "label": r["label"],
        "environment": r["environment"],
        "frames": r["frames"],
        "source_file": r["source_file"],
    }
    for r in recordings
]).sort_values(
    ["environment", "label", "recording_id"]
).reset_index(drop=True)

display(recording_summary)

print("Environment A recordings:", len(recordings_A))
print("Environment B recordings:", len(recordings_B))
print("TOTAL recordings:", len(recordings))

display(
    recording_summary.groupby(
        ["environment", "label"]
    )["recording_id"].count().unstack(fill_value=0)
)

assert len(recordings_A) == 13
assert len(recordings_B) == 12
assert len(recordings) == 25
assert recording_summary["recording_id"].nunique() == 25

## 6. Fixed 5-fold recording-disjoint plan reused for every configuration

In [ ]:
recording_table = recording_summary[
    ["recording_id", "label", "environment"]
].copy().reset_index(drop=True)

outer_cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

fold_plan = []

for fold_index, (development_idx, test_idx) in enumerate(
    outer_cv.split(
        recording_table["recording_id"],
        recording_table["label"],
    ),
    start=1,
):
    development_ids = recording_table.iloc[
        development_idx
    ]["recording_id"].tolist()

    test_ids = recording_table.iloc[
        test_idx
    ]["recording_id"].tolist()

    assert set(development_ids).isdisjoint(test_ids)

    fold_plan.append({
        "fold": fold_index,
        "development_ids": development_ids,
        "test_ids": test_ids,
    })


all_test_ids = [
    rid
    for fold in fold_plan
    for rid in fold["test_ids"]
]

assert len(all_test_ids) == 25
assert len(set(all_test_ids)) == 25
assert set(all_test_ids) == set(recording_table["recording_id"])

plan_rows = []

for fold in fold_plan:
    test_subset = recording_table[
        recording_table["recording_id"].isin(
            fold["test_ids"]
        )
    ]

    plan_rows.append({
        "fold": fold["fold"],
        "development_recordings": len(fold["development_ids"]),
        "test_recordings": len(fold["test_ids"]),
        "test_classes": test_subset["label"].nunique(),
        "test_env_A": (
            test_subset["environment"] == "Environment A"
        ).sum(),
        "test_env_B": (
            test_subset["environment"] == "Environment B"
        ).sum(),
    })

display(pd.DataFrame(plan_rows))

print(
    "Every one of the 33 recordings is used as TEST exactly once — PASSED"
)

## 7. Preprocessing functions

The order matches the historical CARE-BED offline pipeline:

```text
raw TRAIN / VALIDATION / TEST partition
→ optional Hampel
→ optional Savitzky-Golay
→ drop 10 subcarriers
→ segmentation
```

Filtering never crosses a train/validation/test partition boundary.

In [ ]:
def preprocess_partition(
    df,
    use_hampel,
    use_savgol,
):
    processed = pd.DataFrame(
        index=df.index
    )

    for col in df.columns:
        values = df[col].astype(float)

        if use_hampel:
            h = hampel(
                values,
                window_size=HAMPEL_WINDOW,
            )
            values = pd.Series(
                np.asarray(
                    h.filtered_data,
                    dtype=float,
                )
            )

        if use_savgol:
            values = pd.Series(
                savgol_filter(
                    np.asarray(values, dtype=float),
                    window_length=SAVGOL_WINDOW,
                    polyorder=SAVGOL_POLYORDER,
                )
            )

        processed[col] = np.asarray(
            values,
            dtype=float,
        )

    processed.columns = range(
        processed.shape[1]
    )

    return (
        processed
        .drop(
            columns=COLUMNS_TO_DROP,
            axis=1,
        )
        .reset_index(drop=True)
    )


def segment_processed(
    df,
    label,
    recording_id,
    partition,
    window_length,
):
    rows = []

    for start in range(
        0,
        len(df) - window_length + 1,
        STRIDE,
    ):
        end = start + window_length

        rows.append({
            "X": df.iloc[start:end].values.astype(np.float32),
            "label": label,
            "recording_id": recording_id,
            "partition": partition,
            "start": start,
            "end": end,
        })

    return rows

## 8. Precompute full/train/validation partitions for all four filter modes

This is the expensive filtering stage, but it is done only once and cached.

Segmentation at 10/20/40 frames is cheap and will be performed later from these cached processed partitions.

In [ ]:
FILTER_MODES = {
    "NONE": {
        "use_hampel": False,
        "use_savgol": False,
    },
    "H": {
        "use_hampel": True,
        "use_savgol": False,
    },
    "SG": {
        "use_hampel": False,
        "use_savgol": True,
    },
    "HSG": {
        "use_hampel": True,
        "use_savgol": True,
    },
}

cache_signature = {
    "recordings": sorted(
        (
            r["recording_id"],
            len(r["raw"]),
        )
        for r in recordings
    ),
    "train_fraction": DEV_TRAIN_FRACTION,
    "columns_to_drop": COLUMNS_TO_DROP,
    "hampel_window": HAMPEL_WINDOW,
    "savgol_window": SAVGOL_WINDOW,
    "savgol_polyorder": SAVGOL_POLYORDER,
    "version": 1,
}

processed_cache = None

if PREPROCESS_CACHE.exists():
    saved = joblib.load(
        PREPROCESS_CACHE
    )

    if saved.get(
        "signature"
    ) == cache_signature:
        processed_cache = saved[
            "processed_cache"
        ]

        print(
            "Loaded preprocessing cache:",
            PREPROCESS_CACHE,
        )


if processed_cache is None:
    processed_cache = {}
    t0 = time.time()

    for index, recording in enumerate(
        recordings,
        start=1,
    ):
        raw = recording[
            "raw"
        ].reset_index(drop=True)

        cut = int(
            len(raw)
            * DEV_TRAIN_FRACTION
        )

        train_raw = raw.iloc[
            :cut
        ].reset_index(drop=True)

        val_raw = raw.iloc[
            cut:
        ].reset_index(drop=True)

        if (
            len(train_raw) < MIN_FRAMES
            or len(val_raw) < MIN_FRAMES
        ):
            raise ValueError(
                f"{recording['recording_id']}: too short for the 80/20 development split"
            )

        processed_cache[
            recording["recording_id"]
        ] = {
            "label": recording["label"],
            "environment": recording["environment"],
            "modes": {},
        }

        for mode_name, mode in FILTER_MODES.items():
            processed_cache[
                recording["recording_id"]
            ]["modes"][mode_name] = {
                "full": preprocess_partition(
                    raw,
                    mode["use_hampel"],
                    mode["use_savgol"],
                ),
                "train": preprocess_partition(
                    train_raw,
                    mode["use_hampel"],
                    mode["use_savgol"],
                ),
                "val": preprocess_partition(
                    val_raw,
                    mode["use_hampel"],
                    mode["use_savgol"],
                ),
            }

        print(
            f"[{index:02d}/25] {recording['recording_id']} processed in all filter modes"
        )

    joblib.dump(
        {
            "signature": cache_signature,
            "processed_cache": processed_cache,
        },
        PREPROCESS_CACHE,
        compress=3,
    )

    print(
        f"\nFiltering finished in {(time.time() - t0) / 60:.2f} min."
    )

    print(
        "Saved cache:",
        PREPROCESS_CACHE,
    )

assert len(processed_cache) == 25

## 9. Label encoder and BiLSTM

In [ ]:
ALL_CLASSES = sorted(
    recording_summary[
        "label"
    ].unique()
)

label_encoder = LabelEncoder()
label_encoder.fit(
    ALL_CLASSES
)

print(
    "Class order:",
    list(
        label_encoder.classes_
    ),
)


def build_model(
    window_length
):
    model = keras.Sequential([
        keras.Input(
            shape=(
                window_length,
                N_FEATURES,
            )
        ),
        layers.Bidirectional(
            layers.LSTM(64)
        ),
        layers.Dropout(0.5),
        layers.Dense(
            len(
                label_encoder.classes_
            ),
            activation="softmax",
        ),
    ])

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=[
            "accuracy"
        ],
    )

    return model

## 10. Run one configuration across the same five folds

In [ ]:
def filter_mode_name(use_hampel, use_savgol):
    if use_hampel and use_savgol:
        return "HSG"
    if use_hampel:
        return "H"
    if use_savgol:
        return "SG"
    return "NONE"


def make_segments(
    recording_ids,
    partition,
    mode_name,
    window_length,
):
    rows = []

    for recording_id in recording_ids:
        label = processed_cache[recording_id]["label"]

        processed_df = processed_cache[
            recording_id
        ]["modes"][mode_name][partition]

        rows.extend(
            segment_processed(
                processed_df,
                label,
                recording_id,
                partition,
                window_length,
            )
        )

    return rows


def collect_rows(rows):
    X = np.stack(
        [row["X"] for row in rows]
    ).astype(np.float32)

    y = label_encoder.transform(
        [row["label"] for row in rows]
    ).astype(np.int64)

    return X, y


def scale_with(
    scaler,
    X,
    window_length,
):
    z = scaler.transform(
        X.reshape(len(X), -1)
    )

    return z.reshape(
        len(X),
        window_length,
        N_FEATURES,
    ).astype(np.float32)


def run_configuration(config):
    config_id = config["config_id"]
    window_length = config["window"]

    mode_name = filter_mode_name(
        config["use_hampel"],
        config["use_savgol"],
    )

    fold_rows = []
    prediction_tables = []
    histories = {}

    config_start = time.time()

    print("\n" + "#" * 90)
    print(f"CONFIG {config_id}: {config['label']}")
    print("#" * 90)

    for fold in fold_plan:
        fold_index = fold["fold"]
        development_ids = fold["development_ids"]
        test_ids = fold["test_ids"]

        train_rows = make_segments(
            development_ids,
            "train",
            mode_name,
            window_length,
        )

        val_rows = make_segments(
            development_ids,
            "val",
            mode_name,
            window_length,
        )

        test_rows = make_segments(
            test_ids,
            "full",
            mode_name,
            window_length,
        )

        train_val_ids = {
            row["recording_id"]
            for row in train_rows + val_rows
        }

        test_recording_ids = {
            row["recording_id"]
            for row in test_rows
        }

        assert train_val_ids.isdisjoint(
            test_recording_ids
        )

        assert test_recording_ids == set(
            test_ids
        )

        X_train_raw, y_train = collect_rows(
            train_rows
        )
        X_val_raw, y_val = collect_rows(
            val_rows
        )
        X_test_raw, y_test = collect_rows(
            test_rows
        )

        scaler = StandardScaler()

        scaler.fit(
            X_train_raw.reshape(
                len(X_train_raw),
                -1,
            )
        )

        X_train = scale_with(
            scaler,
            X_train_raw,
            window_length,
        )
        X_val = scale_with(
            scaler,
            X_val_raw,
            window_length,
        )
        X_test = scale_with(
            scaler,
            X_test_raw,
            window_length,
        )

        keras.backend.clear_session()

        random.seed(SEED)
        np.random.seed(SEED)
        tf.keras.utils.set_random_seed(SEED)

        model = build_model(
            window_length
        )

        early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=PATIENCE,
            restore_best_weights=True,
        )

        fold_start = time.time()

        history = model.fit(
            X_train,
            y_train,
            validation_data=(
                X_val,
                y_val,
            ),
            epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[early_stopping],
            shuffle=True,
            verbose=0,
        )

        probabilities = model.predict(
            X_test,
            verbose=0,
        )

        y_pred = np.argmax(
            probabilities,
            axis=1,
        )

        accuracy = accuracy_score(
            y_test,
            y_pred,
        )
        macro_f1 = f1_score(
            y_test,
            y_pred,
            average="macro",
            zero_division=0,
        )
        weighted_f1 = f1_score(
            y_test,
            y_pred,
            average="weighted",
            zero_division=0,
        )

        best_epoch = int(
            np.argmin(
                history.history["val_loss"]
            ) + 1
        )

        fold_rows.append({
            "config_id": config_id,
            "configuration": config["label"],
            "window": window_length,
            "hampel": config["use_hampel"],
            "savgol": config["use_savgol"],
            "fold": fold_index,
            "train_windows": len(y_train),
            "validation_windows": len(y_val),
            "test_windows": len(y_test),
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "weighted_f1": weighted_f1,
            "epochs_run": len(
                history.history["loss"]
            ),
            "best_epoch": best_epoch,
            "fold_minutes": (
                time.time() - fold_start
            ) / 60,
        })

        prediction_metadata = pd.DataFrame([
            {
                "recording_id": row["recording_id"],
                "environment": processed_cache[
                    row["recording_id"]
                ]["environment"],
                "true_label": row["label"],
            }
            for row in test_rows
        ])

        prediction_metadata["config_id"] = config_id
        prediction_metadata["configuration"] = config["label"]
        prediction_metadata["fold"] = fold_index
        prediction_metadata["predicted_label"] = (
            label_encoder.inverse_transform(
                y_pred
            )
        )
        prediction_metadata["confidence"] = np.max(
            probabilities,
            axis=1,
        )

        prediction_tables.append(
            prediction_metadata
        )

        histories[fold_index] = {
            key: list(values)
            for key, values
            in history.history.items()
        }

        print(
            f"{config_id} | fold {fold_index}/5 | "
            f"acc={accuracy:.4f} | "
            f"macroF1={macro_f1:.4f} | "
            f"best_epoch={best_epoch}"
        )

    folds_df = pd.DataFrame(
        fold_rows
    )

    predictions_df = pd.concat(
        prediction_tables,
        ignore_index=True,
    )

    y_true = predictions_df[
        "true_label"
    ].to_numpy()

    y_pred = predictions_df[
        "predicted_label"
    ].to_numpy()

    env_metrics = {}

    for environment in [
        "Environment A",
        "Environment B",
    ]:
        subset = predictions_df[
            predictions_df["environment"]
            == environment
        ]

        env_true = subset[
            "true_label"
        ].to_numpy()

        env_pred = subset[
            "predicted_label"
        ].to_numpy()

        suffix = (
            "A"
            if environment == "Environment A"
            else "B"
        )

        env_metrics[
            f"accuracy_{suffix}"
        ] = accuracy_score(
            env_true,
            env_pred,
        )

        env_metrics[
            f"macro_f1_{suffix}"
        ] = f1_score(
            env_true,
            env_pred,
            average="macro",
            zero_division=0,
        )

        env_metrics[
            f"weighted_f1_{suffix}"
        ] = f1_score(
            env_true,
            env_pred,
            average="weighted",
            zero_division=0,
        )

    aggregate = {
        "config_id": config_id,
        "configuration": config["label"],
        "window": window_length,
        "hampel": config["use_hampel"],
        "savgol": config["use_savgol"],
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "weighted_f1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
        "fold_accuracy_mean": folds_df[
            "accuracy"
        ].mean(),
        "fold_accuracy_std": folds_df[
            "accuracy"
        ].std(ddof=1),
        "fold_macro_f1_mean": folds_df[
            "macro_f1"
        ].mean(),
        "fold_macro_f1_std": folds_df[
            "macro_f1"
        ].std(ddof=1),
        "accuracy_A": env_metrics[
            "accuracy_A"
        ],
        "macro_f1_A": env_metrics[
            "macro_f1_A"
        ],
        "weighted_f1_A": env_metrics[
            "weighted_f1_A"
        ],
        "accuracy_B": env_metrics[
            "accuracy_B"
        ],
        "macro_f1_B": env_metrics[
            "macro_f1_B"
        ],
        "weighted_f1_B": env_metrics[
            "weighted_f1_B"
        ],
        "balanced_environment_macro_f1": (
            env_metrics["macro_f1_A"]
            + env_metrics["macro_f1_B"]
        ) / 2,
        "minutes": (
            time.time()
            - config_start
        ) / 60,
    }

    return (
        folds_df,
        predictions_df,
        histories,
        aggregate,
    )

## 11. Run all four W20 filter configurations on the same 5-fold split


In [ ]:
all_fold_tables = []
all_prediction_tables = []
all_histories = {}
summary_rows = []

ablation_start = time.time()

for config_index, config in enumerate(
    CONFIGURATIONS,
    start=1,
):
    folds_df, predictions_df, histories, aggregate = run_configuration(
        config
    )

    all_fold_tables.append(
        folds_df
    )

    all_prediction_tables.append(
        predictions_df
    )

    all_histories[
        config[
            "config_id"
        ]
    ] = histories

    summary_rows.append(
        aggregate
    )

    elapsed = (
        time.time()
        - ablation_start
    ) / 60

    average_config_time = (
        elapsed
        / config_index
    )

    remaining_configs = (
        len(
            CONFIGURATIONS
        )
        - config_index
    )

    print(
        f"Finished {config['config_id']} | "
        f"estimated remaining≈"
        f"{average_config_time * remaining_configs:.1f} min"
    )


all_folds_df = pd.concat(
    all_fold_tables,
    ignore_index=True,
)

all_predictions_df = pd.concat(
    all_prediction_tables,
    ignore_index=True,
)

summary_df = (
    pd.DataFrame(
        summary_rows
    )
    .sort_values(
        [
            "macro_f1",
            "accuracy",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    summary_df
)

all_folds_df.to_csv(
    ALL_FOLDS_CSV,
    index=False,
)

all_predictions_df.to_csv(
    PREDICTIONS_CSV,
    index=False,
)

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
)

environment_summary_df = summary_df[
    [
        "config_id",
        "configuration",
        "accuracy",
        "macro_f1",
        "accuracy_A",
        "macro_f1_A",
        "accuracy_B",
        "macro_f1_B",
        "balanced_environment_macro_f1",
    ]
].copy()

display(environment_summary_df)

environment_summary_df.to_csv(
    ENV_SUMMARY_CSV,
    index=False,
)

## 12. Filter-ablation results — 20-frame input


In [ ]:
filter_ids = [
    "W20_NONE",
    "W20_H",
    "W20_SG",
    "W20_HSG",
]

filter_df = (
    summary_df[
        summary_df[
            "config_id"
        ].isin(
            filter_ids
        )
    ]
    .copy()
)

filter_order = {
    "W20_NONE": 0,
    "W20_H": 1,
    "W20_SG": 2,
    "W20_HSG": 3,
}

filter_df[
    "_order"
] = filter_df[
    "config_id"
].map(
    filter_order
)

filter_df = (
    filter_df
    .sort_values(
        "_order"
    )
    .drop(
        columns=[
            "_order"
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    filter_df[
        [
            "configuration",
            "accuracy",
            "macro_f1",
            "weighted_f1",
            "fold_accuracy_mean",
            "fold_accuracy_std",
        ]
    ]
)

filter_df.to_csv(
    FILTER_CSV,
    index=False,
)


fig, ax = plt.subplots(
    figsize=(9, 5)
)

x = np.arange(
    len(
        filter_df
    )
)

ax.bar(
    x,
    filter_df[
        "accuracy"
    ],
)

ax.set_xticks(
    x
)

ax.set_xticklabels(
    [
        "None",
        "Hampel",
        "SG",
        "Hampel + SG",
    ]
)

ax.set_ylim(
    0,
    1,
)

ax.set_ylabel(
    "Aggregated held-out accuracy"
)

ax.set_xlabel(
    "20-frame preprocessing configuration"
)

ax.set_title(
    "A13 + B12 — recording-disjoint filter ablation"
)

for index, value in enumerate(
    filter_df[
        "accuracy"
    ]
):
    ax.text(
        index,
        value + 0.015,
        f"{value:.3f}",
        ha="center",
    )

fig.tight_layout()

fig.savefig(
    FILTER_PDF,
    bbox_inches="tight",
)

plt.show()

print(
    "Saved:",
    FILTER_PDF,
)

## 13. Package result files


In [ ]:
result_files = [
    ALL_FOLDS_CSV,
    SUMMARY_CSV,
    ENV_SUMMARY_CSV,
    FILTER_CSV,
    PREDICTIONS_CSV,
    FILTER_PDF,
]

with zipfile.ZipFile(
    RESULT_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for file_path in result_files:
        if file_path.exists():
            zf.write(file_path, arcname=file_path.name)

print("Created:", RESULT_ZIP)

try:
    from google.colab import files
    files.download(str(RESULT_ZIP))
except ImportError:
    pass
